In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("NYT_API_KEY")
print(key is not None)

True


In [3]:
import requests
url = "https://api.nytimes.com/svc/books/v3/lists/overview.json"
resp = requests.get(url, params={"api-key": key})
resp.raise_for_status()
resp.status_code

200

In [ ]:
# loading the data from the API and exploring the structure
data = resp.json()
print(type(data))
print(data.keys())
print(data["num_results"])
print(type(data["results"]))
print(data["results"].keys())
print(type(data["results"]["lists"]))
print(len(data["results"]["lists"]))
print(data["results"]["lists"][0])
print(type(data["results"]["lists"][0]["books"]))
print(len(data["results"]["lists"][0]["books"]))
print(data["results"]["lists"][0]["books"][0])
print(type(data["results"]["lists"][0]["books"][0]))
print(data["results"]["lists"][0]["books"][0]["rank"])
print(data["results"]["lists"][0]["books"][0]["title"])
print(data["results"]["lists"][0]["books"][0]["author"])
print(data["results"]["lists"][0]["books"][0]["primary_isbn13"])
print(data["results"]["lists"][0]["books"][0]["publisher"])
print(data["results"]["lists"][0]["books"][0]["rank_last_week"])
print(data["results"]["lists"][0]["books"][0]["weeks_on_list"])
print(data["results"]["lists"][0]["books"][0]["price"])

<class 'dict'>
dict_keys(['status', 'copyright', 'num_results', 'results'])
240
<class 'dict'>
dict_keys(['previous_published_date', 'published_date', 'next_published_date', 'published_date_description', 'bestsellers_date', 'lists', 'monthly_uri', 'weekly_uri'])
<class 'list'>
19
{'display_name': 'Combined Print & E-Book Fiction', 'list_name': 'Combined Print & E-Book Fiction', 'list_name_encoded': 'combined-print-and-e-book-fiction', 'normal_list_ends_at': 15, 'updated': 'WEEKLY', 'list_id': 704, 'uri': 'nyt://bestsellerslist/2dc50b40-8297-5ddb-8a4d-60765725ddaf', 'books': [{'age_group': '', 'amazon_product_url': 'https://www.amazon.com/dp/B0GPJPND4T?tag=thenewyorktim-20', 'article_chapter_link': '', 'asterisk': 0, 'author': 'Janet Evanovich', 'book_image': 'https://static01.nyt.com/bestsellers/images/9781668069738.jpg', 'book_image_height': 500, 'book_image_width': 329, 'book_review_link': '', 'book_uri': 'nyt://book/29fee809-79d9-52b9-8b5d-687b2eeaedd7', 'contributor': 'by Janet Eva

In [ ]:
# checking if number of actual books is same as announced by API
print(sum(len(bs_list["books"]) for bs_list in data["results"]["lists"]))

240


In [ ]:
# loading the data into dataframe and checking for missing ISBNs
import pandas as pd
bs_lists = data["results"]["lists"]
df = pd.json_normalize(data=bs_lists, record_path="books", meta=["list_name", "list_id"])
print(df.shape)
print(df.columns)
df.insert(loc=0, column="bs_published_date", value=data["results"]["published_date"])
print(df.shape)
print(df.columns)
print(df["primary_isbn13"].nunique())
print(df["primary_isbn13"].value_counts())
print(df["primary_isbn13"].isna().sum())
print((df["primary_isbn13"] == "").sum())

(240, 30)
Index(['age_group', 'amazon_product_url', 'article_chapter_link', 'asterisk',
       'author', 'book_image', 'book_image_height', 'book_image_width',
       'book_review_link', 'book_uri', 'contributor', 'contributor_note',
       'created_date', 'dagger', 'description', 'first_chapter_link', 'price',
       'primary_isbn10', 'primary_isbn13', 'publisher', 'rank',
       'rank_last_week', 'sunday_review_link', 'title', 'updated_date',
       'weeks_on_list', 'isbns', 'buy_links', 'list_name', 'list_id'],
      dtype='str')
(240, 31)
Index(['bs_published_date', 'age_group', 'amazon_product_url',
       'article_chapter_link', 'asterisk', 'author', 'book_image',
       'book_image_height', 'book_image_width', 'book_review_link', 'book_uri',
       'contributor', 'contributor_note', 'created_date', 'dagger',
       'description', 'first_chapter_link', 'price', 'primary_isbn10',
       'primary_isbn13', 'publisher', 'rank', 'rank_last_week',
       'sunday_review_link', 'title', 

In [ ]:
# checking how debuts are marked: found that weeks_on_list == 1
last_week_rank_zero = df[df["rank_last_week"] == 0]
print(last_week_rank_zero["weeks_on_list"].value_counts())

weeks_on_list
1      71
3       3
7       1
33      1
29      1
14      1
296     1
105     1
71      1
81      1
20      1
74      1
Name: count, dtype: int64


In [ ]:
weeks_on_list_zero = df[df["weeks_on_list"] == 0]
print(weeks_on_list_zero["rank_last_week"].value_counts())

debuts = df[df["weeks_on_list"] == 1]
print(weeks_on_list_zero["rank_last_week"].value_counts())

Series([], Name: count, dtype: int64)
rank_last_week
0    71
Name: count, dtype: int64
